# How to use rugo with CSV

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mabel-dev/rugo.dev/blob/main/docs/notebooks/csv.ipynb)

Read a CSV file, check its inferred types, read only the columns and rows you need, load it into pandas, and write CSV - with rugo, which needs no PyArrow, pandas or NumPy of its own.

Every example uses the same dataset: 4,630 orbital launch attempts, from Sputnik onwards. More at [rugo.dev](https://rugo.dev).

In [ ]:
%pip install -q rugo pandas

In [ ]:
import os
import urllib.request

if not os.path.exists("space_missions.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/rugo.dev/main/docs/data/space_missions.csv", "space_missions.csv")
print(f"space_missions.csv: {os.path.getsize('space_missions.csv'):,} bytes")

## Read a file

`read_csv` is a context manager that yields a **morsel** - a batch of typed columns. Column types are inferred from a sample of each column's values.

In [ ]:
from rugo import csv

with csv.read_csv("space_missions.csv") as reader:
    for morsel in reader:
        print(morsel)

## Look at the schema

`read_metadata` counts the rows and infers each column's type from the first rows. The first launches in this file have no price, so `Price` shows as text here; `read_csv` samples non-empty values instead, and reads it as a float.

In [ ]:
meta = csv.read_metadata("space_missions.csv")
print(meta.num_rows, "rows")
for column in meta.schema_columns:
    print(column)

## Declare the types you know

Pass `explicit_schema` and a column is parsed straight into that type. The launch time is ISO-8601 text in the file; declare it as a timestamp.

In [ ]:
with csv.read_csv("space_missions.csv", explicit_schema={"Lauched_at": "TIMESTAMP[us]"}) as reader:
    morsel = next(iter(reader))

for name, kind in zip(morsel.column_names, morsel.column_types):
    print(f"{name.decode():16} {kind}")

## Read only some columns

In [ ]:
with csv.read_csv("space_missions.csv", columns=["Company", "Mission", "Price"]) as reader:
    for morsel in reader:
        print(morsel)

## Filter rows as you read

`predicates` is a list of `(column, operator, value)`, applied as the file is parsed. The value must match the column's type - `Price` is a float, so compare it with a float.

In [ ]:
with csv.read_csv(
    "space_missions.csv",
    columns=["Company", "Mission", "Price"],
    predicates=[("Price", ">", 100.0)],
) as reader:
    expensive = next(iter(reader))

print(expensive.num_rows, "launches over $100M")
print(expensive)

## Get Python values

A morsel holds typed columns. `column(name).to_pylist()` turns one into a Python list.

In [ ]:
print(sorted(set(expensive.column("Company").to_pylist())))

## Load a file into pandas

Rugo doesn't need pandas, but it's easy to hand data over: each column's `to_pylist()` gives the values, and a dict of lists is a DataFrame. This little helper does it for every morsel a reader yields.

In [ ]:
import pandas as pd


# Every morsel a rugo reader yields, as one pandas DataFrame.
def to_pandas(reader):
    frames = [
        pd.DataFrame({name.decode(): morsel.column(name).to_pylist() for name in morsel.column_names})
        for morsel in reader
    ]
    return pd.concat(frames, ignore_index=True)

In [ ]:
with csv.read_csv("space_missions.csv", explicit_schema={"Lauched_at": "TIMESTAMP[us]"}) as reader:
    df = to_pandas(reader)

df.head()

## Write a file

`write_csv` turns a morsel into RFC 4180 CSV bytes. Pass `delimiter="\t"` for tab-separated output, and read it back the same way.

In [ ]:
with open("expensive.csv", "wb") as f:
    f.write(csv.write_csv(expensive))

with open("expensive.tsv", "wb") as f:
    f.write(csv.write_csv(expensive, delimiter="\t"))

with csv.read_csv("expensive.tsv", delimiter="\t") as reader:
    print(next(iter(reader)).num_rows, "rows read back from the TSV")

## Convert to Parquet

Rugo's readers and writers share the morsel, so converting is a read and a write - and the Parquet file keeps the types, including the declared timestamp.

In [ ]:
from rugo import parquet

with csv.read_csv("space_missions.csv", explicit_schema={"Lauched_at": "TIMESTAMP[us]"}) as reader:
    with open("space_missions.parquet", "wb") as f:
        with parquet.open_parquet_writer(f.write) as writer:
            for morsel in reader:
                writer.write_row_group(morsel)

for column in parquet.read_metadata("space_missions.parquet").schema_columns:
    print(f"{column.name:16} {column.logical_type}")